Assignment 1 — File Reads Objective Practice reading different file formats into Spark DataFrames. Tasks 1. Read customers.csv  2. Read the same file with: o header o inferSchema 3. Read orders.csv using an explicitly defined schema. 4. Convert customers.csv to Parquet. 5. Read the Parquet file. 6. Read products.csv and display the schema. 7. Read multiple CSV files from a directory. 8. Read multiple Parquet files from a directory. 9. Read JSON data containing customer information. 10. Compare the schemas obtained using: o inferSchema o user-defined schema.

In [2]:
##Tasks 1. Read customers.csv

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    LongType, IntegerType, StringType,
    DoubleType, DateType
)

spark = SparkSession.builder \
    .appName("Assignment_FileReads") \
    .getOrCreate()
customer_df=spark.read.csv("/content/customers.csv")

##2. Read the same file with: header, inferSchema

customer_df=spark.read.option("header","true").option("interSchema","true").csv("/content/customers.csv")
customer_df.show()
customer_df.printSchema()

##3. Read orders.csv using an explicitly defined schema.

orders_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("shipping_city", StringType(), True),
    StructField("shipping_state", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("shipping_cost", DoubleType(), True)
])

orders_df = spark.read.option("header", "true").schema(orders_schema).csv("/content/orders.csv")

orders_df.show()
orders_df.printSchema()
##4. Convert customers.csv to Parquet

customer_df.write.mode('overwrite').parquet("/content/customers_parquet")

## 5. Read the Parquet file
customer_df=spark.read.parquet("/content/customers_parquet")
customer_df.show()
customer_df.printSchema()

##6. Read products.csv and display the schema.

products_df=spark.read.csv("/content/products.csv")
products_df.show()
products_df.printSchema()

##7. Read multiple CSV files from a directory

import os
import shutil

os.makedirs("/content/csv_files", exist_ok=True)

shutil.copy(
    "/content/customers.csv",
    "/content/csv_files/customers.csv"
)

shutil.copy(
    "/content/orders.csv",
    "/content/csv_files/orders.csv"
)

multiple_csv_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/csv_files/*.csv")

multiple_csv_df.show()

##8. Read multiple Parquet files from a directory
multiple_parquet_df = spark.read \
    .parquet("/content/customers_parquet")

multiple_parquet_df.show()
##9Read JSON customer information
customer_df.write.mode('overwrite').json("/content/customers_json")

customer_df = spark.read \
    .json("/content/customers_json")
customer_df.show()
customer_df.printSchema()

##10. Compare inferSchema vs user-defined schema
customers_inferred = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/customers.csv")

customers_inferred.printSchema()

customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("date_of_birth", DateType(), True)
])

customers_defined = spark.read \
    .option("header", "true") \
    .schema(customers_schema) \
    .csv("/content/customers.csv")

customers_defined.printSchema()
print("========== INFERRED SCHEMA ==========")
customers_inferred.printSchema()

print("========== USER-DEFINED SCHEMA ==========")
customers_defined.printSchema()

+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|customer_id|first_name|last_name|          email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|
+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|        101|     Rahul|   Sharma|rahul@gmail.com|9876543210|Bangalore|  Karnataka|  India|       2024-01-15|         Premium|   1995-05-12|
|        102|     Priya|    Reddy|priya@gmail.com|9876543211|Hyderabad|  Telangana|  India|       2024-02-20|        Standard|   1997-08-25|
|        103|      Amit|    Kumar| amit@gmail.com|9876543212|  Chennai| Tamil Nadu|  India|       2024-03-10|           Basic|   1994-11-03|
|        104|     Sneha|    Patel|sneha@gmail.com|9876543213|   Mumbai|Maharashtra|  India|       2024-04-05|         Premium|   1996-02-18|
|        105|

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Assignment 2 — File Writes Using orders DataFrame:

Tasks 1. Write orders into CSV.
 2. Write orders into JSON.
 3. Write orders into Parquet.
 4. Write orders into ORC.
 5. Write orders partitioned by order_status.
 6. Write orders partitioned by: o year o month
 7. Write completed orders into a separate Parquet location.
 8. Write the output using: o overwrite o append
 9. Compare the generated files.
10. Read the generated partitioned data back.

In [3]:
## 1. Write orders into CSV
orders_df.write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv("/content/orders_output")

##2. Write orders into JSON.
orders_df.write.mode("overwrite").json("/content/orders_json")

##3. Write orders into Parquet

orders_df.write.mode("overwrite").parquet("/content/orders_parquet")
##4. Write orders into ORC
orders_df.write.mode("overwrite").orc("/content/orders_orc")
##5. Write orders partitioned by order_status.
orders_df.write \
        .partitionBy("order_status") \
        .mode("overwrite") \
        .parquet("/content/orders_partitioned")
## 6. Write orders partitioned by: o year o month
from pyspark.sql.functions import year, month
orders_df = orders_df.withColumn("year", year("order_date"))
orders_df = orders_df.withColumn("month", month("order_date"))

orders_df.write.partitionBy("year", "month") \
    .mode("overwrite") \
    .parquet("/content/orders_partitioned_year_month")

  ##. Write completed orders into a separate Parquet location.
complete_orders = orders_df.filter(orders_df["order_status"] == "Completed")

complete_orders.write.mode("overwrite").parquet("/content/completed_orders")

##8. Write the output using: o overwrite o append
##overwritemode
orders_df.write.mode("overwrite").parquet("/content/orders_overwrite")
##append
orders_df.write.mode("append").parquet("/content/orders_append")

## 9. Compare the generated files
csv_orders = spark.read \
    .option("header", "true") \
    .csv("/content/orders_output") # Corrected path to the output directory

json_orders = spark.read \
    .json("/content/orders_json")

parquet_orders = spark.read \
    .parquet("/content/orders_parquet")

orc_orders = spark.read \
    .orc("/content/orders_orc")

print("CSV:", csv_orders.count())
print("JSON:", json_orders.count())
print("Parquet:", parquet_orders.count())
print("ORC:", orc_orders.count())

## 10. Read the generated partitioned data back.
partitioned_orders = spark.read \
    .parquet("/content/orders_partitioned")

partitioned_orders.show()

CSV: 20
JSON: 20
Parquet: 20
ORC: 20
+--------+-----------+----------+--------------+-------------+--------------+------------+---------------+-------------+------------+
|order_id|customer_id|order_date|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|order_status|
+--------+-----------+----------+--------------+-------------+--------------+------------+---------------+-------------+------------+
|   10001|        101|2026-01-05|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|   Completed|
|   10003|        103|2026-01-10|    NetBanking|      Chennai|    Tamil Nadu|     15600.0|          750.0|        200.0|   Completed|
|   10006|        101|2026-01-18|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|   Completed|
|   10008|        107|2026-01-22|           UPI|        Kochi|        Kerala|     11300.0|          550.0|        175.0|   Completed|
|   10009|        108|202

Assignment 3 — withColumn() Using orders:

1. Create net_amount: total_amount - discount_amount 2. Create final_amount: net_amount + shipping_cost 3. Create discount_percentage. 4. Create order_year. 5. Create order_month. 6. Create order_quarter. 7. Create order_week. 8. Create is_high_value_order. 9. Create order_category: < 500       → Low 500-2000    → Medium > 2000      → High 10. Create order_age_days based on today's date.

In [5]:
##1.Create net_amount: total_amount - discount_amount
from pyspark.sql.functions import col , when
orders_df=orders_df.withColumn("net_amount",col("total_amount")-col("discount_amount"))
orders_df.show()

##2. Create final_amount: net_amount + shipping_cost
orders_df=orders_df.withColumn("final_amount",col("net_amount")+col("shipping_cost"))
orders_df.show()

##3. Create discount_percentage.
orders_df=orders_df.withColumn("discount_percentage",(col("discount_amount")/col("total_amount"))*100)
orders_df.show()

## 4. Create order_year.
from pyspark.sql.functions import year
orders_df=orders_df.withColumn("order_year",year(col("order_date")))
orders_df.show()

##5. Create order_month.
from pyspark.sql.functions import month
orders_df=orders_df.withColumn("order_month",month(col("order_date")))
orders_df.show()

##6. Create order_quarter.
from pyspark.sql.functions import quarter
orders_df=orders_df.withColumn("order_quarter",quarter(col("order_date")))
orders_df.show()

##7. Create order_week.
from pyspark.sql.functions import weekofyear
orders_df=orders_df.withColumn("order_week",weekofyear(col("order_date")))
orders_df.show()

##8. Create is_high_value_order.
orders_df=orders_df.withColumn("is_high_value_order",col("total_amount")>2000)
orders_df.show()

## 9. Create order_category: < 500 → Low 500-2000 → Medium > 2000 → High
orders_df=orders_df.withColumn("order_category",when(col("total_amount")<500,"Low")
                                .when((col("total_amount")>=500) & (col("total_amount")<=2000),"Medium")
                                .otherwise("High"))
orders_df.show()

##10. Create order_age_days based on today's date.
from pyspark.sql.functions import datediff,current_date
orders_df=orders_df.withColumn("order_age_days",datediff(current_date(),col("order_date")))
orders_df.show()


+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----+-----+----------+------------+-------------------+----------+-----------+-------------+----------+-------------------+--------------+--------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|year|month|net_amount|final_amount|discount_percentage|order_year|order_month|order_quarter|order_week|is_high_value_order|order_category|order_age_days|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----+-----+----------+------------+-------------------+----------+-----------+-------------+----------+-------------------+--------------+--------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|2026| 

Assignment 4 — withColumnRenamed()

Using customers: Rename: customer_id       first_name       last_name         → cust_id  → fname → lname registration_date → signup_date customer_segment  → segment

In [6]:
##1 customer_id --> cust_id
from pyspark.sql.functions import col
customer_df=customer_df.withColumnRenamed("customer_id","cust_id")
customer_df.show()
##2 first_name--> fname
customer_df=customer_df.withColumnRenamed("first_name","fname")
customer_df.show()
##3 last_name--> lname
customer_df=customer_df.withColumnRenamed("last_name","lname")
customer_df.show()
##4  registration_date → signup_date
customer_df=customer_df.withColumnRenamed("registration_date","signup_date")
customer_df.show()
##5 customer_segment  → segment
customer_df=customer_df.withColumnRenamed("customer_segment","segment")
customer_df.show()

+---------+-------+-------+----------------+-------------+---------------+----------+---------+----------+-----------------+-----------+
|     city|country|cust_id|customer_segment|date_of_birth|          email|first_name|last_name|     phone|registration_date|      state|
+---------+-------+-------+----------------+-------------+---------------+----------+---------+----------+-----------------+-----------+
|Bangalore|  India|    101|         Premium|   1995-05-12|rahul@gmail.com|     Rahul|   Sharma|9876543210|       2024-01-15|  Karnataka|
|Hyderabad|  India|    102|        Standard|   1997-08-25|priya@gmail.com|     Priya|    Reddy|9876543211|       2024-02-20|  Telangana|
|  Chennai|  India|    103|           Basic|   1994-11-03| amit@gmail.com|      Amit|    Kumar|9876543212|       2024-03-10| Tamil Nadu|
|   Mumbai|  India|    104|         Premium|   1996-02-18|sneha@gmail.com|     Sneha|    Patel|9876543213|       2024-04-05|Maharashtra|
|    Delhi|  India|    105|        Standa

Assignment 5 — Filters

Using orders: Tasks Find: 1. Orders greater than ₹5,000. 2. Completed orders. 3. Cancelled orders. 4. Orders from Karnataka. 5. Orders from Karnataka or Maharashtra. 6. Orders between ₹1,000 and ₹10,000. 7. Orders placed during 2025. 8. Orders where discount is greater than 20%. 9. Orders where payment method is UPI. 10. Orders where status is not Cancelled. 11. Orders where shipping city is NULL. 12. Orders where amount is NOT NULL. 13. Orders satisfying multiple conditions. 14. Find the top-value completed orders where: total_amount > 5000 AND discount_amount > 500 AND payment_method = 'UPI'

In [7]:
##1. Orders greater than ₹5,000
from pyspark.sql.functions import col, year

# Ensure discount_percentage is available on orders_df for filtering tasks
orders_df = orders_df.withColumn("discount_percentage", (col("discount_amount") / col("total_amount")) * 100)

order_df=orders_df.filter(col("total_amount")>5000)
order_df.show()

##2. Completed orders.
order_df=orders_df.filter(col("order_status")=="Completed")
order_df.show()

##. 3. Cancelled orders
order_df=orders_df.filter(col("order_status")=="Cancelled")
order_df.show()

##4.Orders from Karnataka.
order_df=orders_df.filter(col("shipping_state")=="Karnataka")
order_df.show()

#5.5. Orders from Karnataka or Maharashtra
order_df=orders_df.filter((col("shipping_state")=="Karnataka")|(col("shipping_state")=="Maharashtra"))
order_df.show()

#6. Orders between ₹1,000 and ₹10,000.
order_df=orders_df.filter((col("total_amount")>=1000)&(col("total_amount")<=10000))
order_df.show()
#7. Orders placed during 2025
order_df=orders_df.filter(year(col("order_date")) == 2025)
order_df.show()

## 8. Orders where discount is greater than 20%
order_df=orders_df.filter(col("discount_percentage")>20)
order_df.show()

#9. 9. Orders where payment method is UPI
order_df=orders_df.filter(col("payment_method")=="UPI")
order_df.show()

##10. Orders where status is not Cancelled
order_df=orders_df.filter(col("order_status")!="Cancelled")
order_df.show()

##11. Orders where shipping city is NULL
order_df=orders_df.filter(col("shipping_city").isNull())
order_df.show()

##12. Orders where amount is NOT NULL
order_df=orders_df.filter(col("total_amount").isNotNull())
order_df.show()

##13.. Orders satisfying multiple conditions.
order_df=orders_df.filter((col("order_status") == "Completed") & (col("total_amount") > 10000) & (col("shipping_state") == "Karnataka"))
order_df.show()

##14. Find the top-value completed orders where: total_amount > 5000 AND discount_amount > 500 AND payment_method = 'UPI'
order_df=orders_df.filter((col("total_amount")>5000)&(col("discount_amount")>500)&(col("payment_method")=="UPI"))
order_df.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----+-----+----------+------------+-------------------+----------+-----------+-------------+----------+-------------------+--------------+--------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|year|month|net_amount|final_amount|discount_percentage|order_year|order_month|order_quarter|order_week|is_high_value_order|order_category|order_age_days|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----+-----+----------+------------+-------------------+----------+-----------+-------------+----------+-------------------+--------------+--------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|2026| 

Assignment 6 — Aggregation Functions

Using orders: Tasks Calculate: 1. Total number of orders. 2. Total revenue. 3. Average order value. 4. Minimum order value. 5. Maximum order value. 6. Total discount. 7. Total shipping cost. 8. Number of orders by status. 9. Revenue by state. 10. Revenue by payment method. 11. Average order value by customer. 12. Maximum order value by customer. 13. Monthly revenue. 14. Monthly order count. 15. Revenue by state and payment method.

In [8]:
from pyspark.sql.functions import count, sum, avg, min, max, month, col
from pyspark.sql.types import (
    StructType, StructField,
    LongType, IntegerType, StringType,
    DoubleType, DateType
)

# Re-initialize orders_df for this assignment to ensure all original columns are present
# Define the schema again to ensure consistency. SparkSession 'spark' is assumed to be available.
orders_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("shipping_city", StringType(), True),
    StructField("shipping_state", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("shipping_cost", DoubleType(), True)
])

initial_orders_df = spark.read.option("header", "true").schema(orders_schema).csv("/content/orders.csv")

##1. Total number of orders
total_orders_df = initial_orders_df.select(count("order_id").alias("total_numberof_orders"))
total_orders_df.show()

## 2. Total revenue
total_revenue_df = initial_orders_df.select(sum("total_amount").alias("total_revenue"))
total_revenue_df.show()

##3. Average order value.
average_value_df = initial_orders_df.select(avg("total_amount").alias("average_value"))
average_value_df.show()

##4. Minimum order value.
minimum_value_df = initial_orders_df.select(min("total_amount").alias("minimum_value"))
minimum_value_df.show()

## 5. Maximum order value.
maximum_value_df = initial_orders_df.select(max("total_amount").alias("maximum_value"))
maximum_value_df.show()

##6. Total discount.
total_discount_df = initial_orders_df.select(sum("discount_amount").alias("total_discount"))
total_discount_df.show()

## 7. Total shipping cost.
total_shipping_cost_df = initial_orders_df.select(sum("shipping_cost").alias("total_shipping_cost"))
total_shipping_cost_df.show()

##8. Number of orders by status.
orders_by_status_df = initial_orders_df.groupBy("order_status").agg(count("order_id").alias("number_of_orders"))
orders_by_status_df.show()

##9. Revenue by state
revenue_by_state_df = initial_orders_df.groupBy("shipping_state").agg(sum("total_amount").alias("revenue_by_state"))
revenue_by_state_df.show()

##10. Revenue by payment method.
revenue_by_payment_method_df = initial_orders_df.groupBy("payment_method").agg(sum("total_amount").alias("revenue_by_payment_method"))
revenue_by_payment_method_df.show()

##11. Average order value by customer.
avg_order_value_by_customer_df = initial_orders_df.groupBy("customer_id").agg(avg("total_amount").alias("average_order_value"))
avg_order_value_by_customer_df.show()

##2. Maximum order value by customer.
max_order_value_by_customer_df = initial_orders_df.groupBy("customer_id").agg(max("total_amount").alias("maximum_order_value"))
max_order_value_by_customer_df.show()

##13. Monthly revenue.
monthly_revenue_df = initial_orders_df.withColumn("order_month",month(col("order_date"))).groupBy("order_month").agg(sum("total_amount").alias("monthly_revenue"))
monthly_revenue_df.show()

##14. Monthly order count.
monthly_order_count_df = initial_orders_df.withColumn("order_month",month(col("order_date"))).groupBy("order_month").agg(count("order_id").alias("monthly_order_count"))
monthly_order_count_df.show()

##15. Revenue by state and payment method.
revenue_by_state_payment_df = initial_orders_df.groupBy("shipping_state","payment_method").agg(sum("total_amount").alias("revenue_by_state_payment"))
revenue_by_state_payment_df.show()

+---------------------+
|total_numberof_orders|
+---------------------+
|                   20|
+---------------------+

+-------------+
|total_revenue|
+-------------+
|     251450.0|
+-------------+

+-------------+
|average_value|
+-------------+
|      12572.5|
+-------------+

+-------------+
|minimum_value|
+-------------+
|       4500.0|
+-------------+

+-------------+
|maximum_value|
+-------------+
|      24500.0|
+-------------+

+--------------+
|total_discount|
+--------------+
|       11350.0|
+--------------+

+-------------------+
|total_shipping_cost|
+-------------------+
|             3520.0|
+-------------------+

+------------+----------------+
|order_status|number_of_orders|
+------------+----------------+
|    Returned|               2|
|   Completed|              11|
|   Cancelled|               3|
|     Pending|               4|
+------------+----------------+

+--------------+----------------+
|shipping_state|revenue_by_state|
+--------------+----------------+

Assignment 7 — Window Functions

Using orders: Tasks 1. Find the customer's first order. 2. Find the customer's latest order. 3. Rank customers based on total spending. 4. Rank orders within each customer. 5. Find the top 3 orders for every customer. 6. Calculate running revenue for each customer. 7. Calculate previous order amount using lag(). 8. Calculate next order amount using lead(). 9. Calculate difference between current and previous order. 10. Calculate cumulative order count for each customer. Bonus Find customers whose latest order value is greater than their previous order.

In [9]:
##1. Find the customer's first order
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/orders.csv")

orders_df.printSchema()
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, first
window_spec = Window.partitionBy("customer_id") \
    .orderBy(col("order_date").asc(), col("order_id").asc())

first_order_df = orders_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

first_order_df.show()

##2. Find the customer's latest order
window_spec = Window.partitionBy("customer_id") \
    .orderBy(col("order_date").desc(), col("order_id").desc())

latest_order_df = orders_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

latest_order_df.show()

##3. Rank customers based on total spending
customer_spending_df = orders_df.groupBy("customer_id") \
    .agg(sum("total_amount").alias("total_spending"))
rank_window = Window.orderBy(col("total_spending").desc())
customer_spending_df = customer_spending_df \
    .withColumn("rank", row_number().over(rank_window))
customer_spending_df.show()

##4. Rank orders within each customer
from pyspark.sql.functions import rank
window_spec = Window.partitionBy("customer_id") .orderBy(col("total_amount").desc())

ranked_orders_df = orders_df.withColumn("order_rank",rank().over(window_spec))

ranked_orders_df.show()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- shipping_state: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- shipping_cost: double (nullable = true)

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|
|   10002|      

In [10]:
##5 Find the top 3 orders for every customer
window_spec = Window.partitionBy("customer_id").orderBy(col("total_amount").desc())

top_3_orders_df = orders_df.withColumn("order_rank",row_number().over(window_spec)).filter(col("order_rank") <= 3)

top_3_orders_df.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|order_rank|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----------+
|   10006|        101|2026-01-18|   Completed|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|         1|
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|         2|
|   10002|        102|2026-01-07|     Pending|           UPI|    Hyderabad|     Telangana|      8200.0|          250.0|        100.0|         1|
|   10003|        103|2026-01-10|   Completed|    NetBanking|      Chennai|    Tamil Nadu|     15600.0|          750.0|        200

In [11]:
## 6. Calculate running revenue for each customer.
window_spec = Window.partitionBy("customer_id").orderBy(col("order_date").asc(),col("order_id").asc())\
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )

running_revenue_df = orders_df.withColumn("running_revenue",sum("total_amount").over(window_spec))
running_revenue_df.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|running_revenue|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|        12500.0|
|   10006|        101|2026-01-18|   Completed|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|        34500.0|
|   10002|        102|2026-01-07|     Pending|           UPI|    Hyderabad|     Telangana|      8200.0|          250.0|        100.0|         8200.0|
|   10003|        103|2026-01-10|   Completed|    NetBanking|      Chennai|    Tamil Nadu|     15600

7. Calculate previous order amount using lag()


In [12]:
from pyspark.sql.functions import lag
window_spec = Window.partitionBy("customer_id").orderBy(col("order_date").asc(),col("order_id").asc())

previous_order_df = orders_df.withColumn("previous_order_amount",lag("total_amount").over(window_spec))

previous_order_df.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|previous_order_amount|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|                 NULL|
|   10006|        101|2026-01-18|   Completed|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|              12500.0|
|   10002|        102|2026-01-07|     Pending|           UPI|    Hyderabad|     Telangana|      8200.0|          250.0|        100.0|                 NULL|
|   10003|        103|2026-01-10|   Completed|    NetBanking|   

8. Calculate next order amount using lead()

In [13]:
from pyspark.sql.functions import lead
window_spec = Window.partitionBy("customer_id").orderBy(col("order_date").asc(),col("order_id").asc())

next_order_df = orders_df.withColumn("next_order_amount",lead("total_amount").over(window_spec))

next_order_df.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+-----------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|next_order_amount|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+-----------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|          22000.0|
|   10006|        101|2026-01-18|   Completed|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|             NULL|
|   10002|        102|2026-01-07|     Pending|           UPI|    Hyderabad|     Telangana|      8200.0|          250.0|        100.0|             NULL|
|   10003|        103|2026-01-10|   Completed|    NetBanking|      Chennai|    Tamil Nad

9. Calculate difference between current and previous order

In [14]:
window_spec = Window.partitionBy("customer_id").orderBy(col("order_date").asc(),col("order_id").asc())

difference_df = orders_df \
    .withColumn(
        "previous_order_amount",
        lag("total_amount").over(window_spec)
    ) \
    .withColumn(
        "order_difference",
        col("total_amount") - col("previous_order_amount")
    )

difference_df.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------------+----------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|previous_order_amount|order_difference|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------------+----------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|                 NULL|            NULL|
|   10006|        101|2026-01-18|   Completed|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|              12500.0|          9500.0|
|   10002|        102|2026-01-07|     Pending|           UPI|    Hyderabad|     Telangana|      8200.0|          250.0|        100.0|  

10. Calculate cumulative order count for each customer

In [15]:
window_spec = Window.partitionBy("customer_id").orderBy(col("order_date").asc(),col("order_id").asc()
    ) \
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )

cumulative_count_df = orders_df \
    .withColumn(
        "cumulative_order_count",
        count("*").over(window_spec)
    )

cumulative_count_df.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----------------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|cumulative_order_count|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----------------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|                     1|
|   10006|        101|2026-01-18|   Completed|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|                     2|
|   10002|        102|2026-01-07|     Pending|           UPI|    Hyderabad|     Telangana|      8200.0|          250.0|        100.0|                     1|
|   10003|        103|2026-01-10|   Completed|    NetBanki

11.Find customers whose latest order value is greater than their previous order.

In [16]:
window_spec = Window.partitionBy("customer_id").orderBy(col("order_date").desc(),col("order_id").desc())

bonus_df = orders_df.withColumn("previous_order_amount",lag("total_amount").over(window_spec)) \
    .withColumn("row_num",row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .filter(col("total_amount") > col("previous_order_amount")) \
    .drop("row_num")

bonus_df.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|previous_order_amount|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------------+
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------------+



**Assignment 8 — Joins **

Use: customers orders order_items products payments Tasks Implement and understand: 1. Inner join 2. Left join 3. Right join 4. Full outer join 5. Left semi join 6. Left anti join 7. Cross join 8. Self join Questions 1. Find customers who placed orders. 2. Find customers who never placed an order. 3. Find products that were never sold. 4. Find orders without payment records. 5. Find customers with successful payments. 6. Find all customers and their orders. 7. Find employees and their managers using self join. 8. Create an order-product dataset by joining: o orders o order_items o products

1.. Find customers who placed orders.

In [17]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Assignment8") \
    .getOrCreate()
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/customers.csv")
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/orders.csv")

customers_with_orders = customers_df.join(
    orders_df,
    customers_df.customer_id == orders_df.customer_id,
    "left_semi"
)

customers_with_orders.show()

+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|customer_id|first_name|last_name|          email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|
+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|        101|     Rahul|   Sharma|rahul@gmail.com|9876543210|Bangalore|  Karnataka|  India|       2024-01-15|         Premium|   1995-05-12|
|        102|     Priya|    Reddy|priya@gmail.com|9876543211|Hyderabad|  Telangana|  India|       2024-02-20|        Standard|   1997-08-25|
|        103|      Amit|    Kumar| amit@gmail.com|9876543212|  Chennai| Tamil Nadu|  India|       2024-03-10|           Basic|   1994-11-03|
|        104|     Sneha|    Patel|sneha@gmail.com|9876543213|   Mumbai|Maharashtra|  India|       2024-04-05|         Premium|   1996-02-18|
|        105|

2.Find customers who never placed an order

In [18]:
customers_never_ordered = customers_df.join(
    orders_df.select("customer_id").distinct(),"customer_id","left_anti")
customers_never_ordered.show()

+-----------+----------+---------+-----+-----+----+-----+-------+-----------------+----------------+-------------+
|customer_id|first_name|last_name|email|phone|city|state|country|registration_date|customer_segment|date_of_birth|
+-----------+----------+---------+-----+-----+----+-----+-------+-----------------+----------------+-------------+
+-----------+----------+---------+-----+-----+----+-----+-------+-----------------+----------------+-------------+



3.Find products that were never sold

In [20]:
products_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/products.csv")
order_items_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/order_items.csv")

products_never_sold = products_df.join(
    order_items_df.select("product_id").distinct(),
    products_df.product_id == order_items_df.product_id,
    "left_anti")
products_never_sold.show()

+----------+------------+--------+------------+-----+-----------+---------+-------------+-----------+--------------+--------------+
|product_id|product_name|category|sub_category|brand|supplier_id|unit_cost|selling_price|launch_date|product_rating|stock_quantity|
+----------+------------+--------+------------+-----+-----------+---------+-------------+-----------+--------------+--------------+
+----------+------------+--------+------------+-----+-----------+---------+-------------+-----------+--------------+--------------+



Q4. Find orders without payment records.

In [22]:
payments_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/payments.csv")
orders_without_payment = orders_df.join(
    payments_df.select("order_id").distinct(),
    orders_df.order_id == payments_df.order_id,
    "left_anti")

orders_without_payment.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+



Q5. Find customers with successful payments.

In [24]:
successful_payments = payments_df.filter(payments_df.payment_status == 'Completed')

customers_successful_payment = (customers_df
    .join(
        orders_df,
        customers_df.customer_id == orders_df.customer_id,
        "inner"
    )
    .join(
        successful_payments.select("order_id").distinct(),
        orders_df.order_id == successful_payments.order_id,
        "inner"
    )
    .select(customers_df["*"])
    .distinct()
)

customers_successful_payment.show()

+-----------+----------+---------+-----+-----+----+-----+-------+-----------------+----------------+-------------+
|customer_id|first_name|last_name|email|phone|city|state|country|registration_date|customer_segment|date_of_birth|
+-----------+----------+---------+-----+-----+----+-----+-------+-----------------+----------------+-------------+
+-----------+----------+---------+-----+-----+----+-----+-------+-----------------+----------------+-------------+



Q6. Find all customers and their orders.

In [25]:
all_customers_orders = customers_df.join(
    orders_df,
    customers_df.customer_id == orders_df.customer_id,
    "left")
all_customers_orders.show()

+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|customer_id|first_name|last_name|          email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|        101|     Rahul|   Sharma|rahul@gmail.com|9876543210|Bangalore|  Karnataka|  India|       2024-01-15|         Premium|   1995-05-12|   10006|        101|2026-01-18|   Comple

Q7. Find employees and their managers using self join.

In [29]:
employees_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/employees.csv")

# Pre-select and rename columns for the 'employee' side of the join
employees = employees_df.select(
    col("employee_id").alias("e_employee_id"),
    col("employee_name").alias("e_employee_name"),
    col("department").alias("e_department"),
    col("manager_id").alias("e_manager_id")
)

# Pre-select and rename columns for the 'manager' side of the join
managers = employees_df.select(
    col("employee_id").alias("m_employee_id"),
    col("employee_name").alias("m_employee_name")
)

employee_manager_df = employees.join(
    managers,
    employees.e_manager_id == managers.m_employee_id, # Use the new prefixed column names for join condition
    "left"
).select(
    col("e_employee_id").alias("employee_id"),
    col("e_employee_name").alias("employee_name"),
    col("e_department").alias("department"),
    col("e_manager_id").alias("manager_id"), # Renaming back to original for output, but now unambiguous
    col("m_employee_name").alias("manager_name")
)

employee_manager_df.show()

+-----------+--------------+----------------+----------+------------+
|employee_id| employee_name|      department|manager_id|manager_name|
+-----------+--------------+----------------+----------+------------+
|          1|   Arjun Kumar|     Engineering|      NULL|        NULL|
|          2|  Priya Sharma|Data Engineering|       1.0| Arjun Kumar|
|          3|   Rahul Verma|Data Engineering|       1.0| Arjun Kumar|
|          4|   Sneha Reddy|       Analytics|       5.0|Vikram Singh|
|          5|  Vikram Singh|       Analytics|      NULL|        NULL|
|          6|    Neha Patel|         Finance|       9.0|  Meera Nair|
|          7|     Kiran Rao|         Finance|       9.0|  Meera Nair|
|          8|    Amit Joshi|              HR|      12.0| Rohit Menon|
|          9|    Meera Nair|         Finance|      NULL|        NULL|
|         10|  Sanjay Gupta|     Engineering|       1.0| Arjun Kumar|
|         11|    Divya Iyer|Data Engineering|       1.0| Arjun Kumar|
|         12|   Rohi

Q8. Create an order-product dataset

In [30]:
order_product_df = (
    orders_df
    .join(
        order_items_df,
        orders_df.order_id == order_items_df.order_id,
        "inner"
    )
    .join(
        products_df,
        order_items_df.product_id == products_df.product_id,
        "inner"
    )
)

order_product_df.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+-------------+--------+----------+--------+----------+----------------+-----------+----------+--------------------+---------------+------------+-----------+-----------+---------+-------------+-----------+--------------+--------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|order_item_id|order_id|product_id|quantity|unit_price|discount_percent|item_amount|product_id|        product_name|       category|sub_category|      brand|supplier_id|unit_cost|selling_price|launch_date|product_rating|stock_quantity|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+-------------+--------+----------+--------+----------+----------------+-----------+----------+--------------------+---------------+

**Asignment 9**

orderBy


Q1. Sort orders by amount ascending

In [33]:
orders_sorted_asc = orders_df.orderBy(col("total_amount").asc())
orders_sorted_asc.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10004|        104|2026-01-12|   Cancelled|          Card|       Mumbai|   Maharashtra|      4500.0|          200.0|        120.0|
|   10010|        109|2026-01-28|   Cancelled|          Card|    Ahmedabad|       Gujarat|      5300.0|          250.0|        130.0|
|   10016|        115|2026-02-17|   Completed|          Cash|      Lucknow| Uttar Pradesh|      6100.0|          300.0|        125.0|
|   10007|        106|2026-01-20|     Pending|          Cash|        Delhi|         Delhi|      6700.0|          300.0|        150.0|
|   10019|        118|2026-02-26|   Cancelled|    NetBanking| 

Q2. Sort orders by amount descending

In [35]:
orders_sorted_desc = orders_df.orderBy(col("total_amount").desc())

orders_sorted_desc.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10018|        117|2026-02-23|   Completed|          Card|       Mumbai|   Maharashtra|     24500.0|         1200.0|        275.0|
|   10006|        101|2026-01-18|   Completed|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|
|   10013|        112|2026-02-08|   Completed|    NetBanking|      Kolkata|   West Bengal|     20100.0|          850.0|        210.0|
|   10009|        108|2026-01-25|   Completed|    NetBanking|       Jaipur|     Rajasthan|     18900.0|          900.0|        225.0|
|   10017|        116|2026-02-20|     Pending|           UPI| 

Q3. Sort by date descending

In [36]:
orders_by_date = orders_df.orderBy( col("order_date").desc())

orders_by_date.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10020|        119|2026-03-01|   Completed|           UPI|      Chennai|    Tamil Nadu|     16800.0|          800.0|        220.0|
|   10019|        118|2026-02-26|   Cancelled|    NetBanking|         Pune|   Maharashtra|      7200.0|          350.0|        150.0|
|   10018|        117|2026-02-23|   Completed|          Card|       Mumbai|   Maharashtra|     24500.0|         1200.0|        275.0|
|   10017|        116|2026-02-20|     Pending|           UPI|    Bengaluru|     Karnataka|     17500.0|          700.0|        200.0|
|   10016|        115|2026-02-17|   Completed|          Cash| 

Q4. Sort by state and amount

In [38]:
orders_by_state_amount = orders_df.orderBy(col("shipping_state").asc(),col("total_amount").asc())

orders_by_state_amount.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10014|        113|2026-02-11|    Returned|           UPI|   Chandigarh|    Chandigarh|      8900.0|          450.0|        170.0|
|   10007|        106|2026-01-20|     Pending|          Cash|        Delhi|         Delhi|      6700.0|          300.0|        150.0|
|   10010|        109|2026-01-28|   Cancelled|          Card|    Ahmedabad|       Gujarat|      5300.0|          250.0|        130.0|
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|
|   10011|        110|2026-02-02|   Completed|           UPI| 

Q5. Sort by status ascending and amount descending

In [40]:
orders_by_status_amount = orders_df.orderBy(col("order_status").asc(),col("total_amount").desc())

orders_by_status_amount.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10019|        118|2026-02-26|   Cancelled|    NetBanking|         Pune|   Maharashtra|      7200.0|          350.0|        150.0|
|   10010|        109|2026-01-28|   Cancelled|          Card|    Ahmedabad|       Gujarat|      5300.0|          250.0|        130.0|
|   10004|        104|2026-01-12|   Cancelled|          Card|       Mumbai|   Maharashtra|      4500.0|          200.0|        120.0|
|   10018|        117|2026-02-23|   Completed|          Card|       Mumbai|   Maharashtra|     24500.0|         1200.0|        275.0|
|   10006|        101|2026-01-18|   Completed|          Card| 

Q6. Sort NULL values first

In [43]:
orders_null_first = orders_df.orderBy(col("total_amount").asc_nulls_first())

orders_null_first.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10004|        104|2026-01-12|   Cancelled|          Card|       Mumbai|   Maharashtra|      4500.0|          200.0|        120.0|
|   10010|        109|2026-01-28|   Cancelled|          Card|    Ahmedabad|       Gujarat|      5300.0|          250.0|        130.0|
|   10016|        115|2026-02-17|   Completed|          Cash|      Lucknow| Uttar Pradesh|      6100.0|          300.0|        125.0|
|   10007|        106|2026-01-20|     Pending|          Cash|        Delhi|         Delhi|      6700.0|          300.0|        150.0|
|   10019|        118|2026-02-26|   Cancelled|    NetBanking| 

Q7. Sort NULL values last

In [44]:
orders_null_last = orders_df.orderBy(col("total_amount").asc_nulls_last())

orders_null_last.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10004|        104|2026-01-12|   Cancelled|          Card|       Mumbai|   Maharashtra|      4500.0|          200.0|        120.0|
|   10010|        109|2026-01-28|   Cancelled|          Card|    Ahmedabad|       Gujarat|      5300.0|          250.0|        130.0|
|   10016|        115|2026-02-17|   Completed|          Cash|      Lucknow| Uttar Pradesh|      6100.0|          300.0|        125.0|
|   10007|        106|2026-01-20|     Pending|          Cash|        Delhi|         Delhi|      6700.0|          300.0|        150.0|
|   10019|        118|2026-02-26|   Cancelled|    NetBanking| 

Q8. Find the top 100 highest-value orders

In [47]:
top_100_orders = orders_df.orderBy(col("total_amount").desc()).limit(100)

top_100_orders.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10018|        117|2026-02-23|   Completed|          Card|       Mumbai|   Maharashtra|     24500.0|         1200.0|        275.0|
|   10006|        101|2026-01-18|   Completed|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|
|   10013|        112|2026-02-08|   Completed|    NetBanking|      Kolkata|   West Bengal|     20100.0|          850.0|        210.0|
|   10009|        108|2026-01-25|   Completed|    NetBanking|       Jaipur|     Rajasthan|     18900.0|          900.0|        225.0|
|   10017|        116|2026-02-20|     Pending|           UPI| 

Q9. Find the 100 oldest orders

In [48]:
oldest_100_orders = orders_df.orderBy(col("order_date").asc()).limit(100)

oldest_100_orders.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|
|   10002|        102|2026-01-07|     Pending|           UPI|    Hyderabad|     Telangana|      8200.0|          250.0|        100.0|
|   10003|        103|2026-01-10|   Completed|    NetBanking|      Chennai|    Tamil Nadu|     15600.0|          750.0|        200.0|
|   10004|        104|2026-01-12|   Cancelled|          Card|       Mumbai|   Maharashtra|      4500.0|          200.0|        120.0|
|   10005|        105|2026-01-15|    Returned|           UPI| 

Q10. Sort customers based on total spending

In [50]:
from pyspark.sql.functions import sum, col

customer_spending = (
    orders_df
    .groupBy("customer_id")
    .agg(
        sum("total_amount").alias("total_spending")
    )
    .orderBy(
        col("total_spending").desc()
    )
)

customer_spending.show()

+-----------+--------------+
|customer_id|total_spending|
+-----------+--------------+
|        101|       34500.0|
|        117|       24500.0|
|        112|       20100.0|
|        108|       18900.0|
|        116|       17500.0|
|        119|       16800.0|
|        103|       15600.0|
|        110|       14750.0|
|        114|       13200.0|
|        107|       11300.0|
|        105|        9800.0|
|        113|        8900.0|
|        102|        8200.0|
|        111|        7600.0|
|        118|        7200.0|
|        106|        6700.0|
|        115|        6100.0|
|        109|        5300.0|
|        104|        4500.0|
+-----------+--------------+



**Assignment 10** Data functions

Q1. Extract year from order date

In [51]:
q1 = orders_df.select("order_id","order_date",year("order_date").alias("order_year"))

q1.show()

+--------+----------+----------+
|order_id|order_date|order_year|
+--------+----------+----------+
|   10001|2026-01-05|      2026|
|   10002|2026-01-07|      2026|
|   10003|2026-01-10|      2026|
|   10004|2026-01-12|      2026|
|   10005|2026-01-15|      2026|
|   10006|2026-01-18|      2026|
|   10007|2026-01-20|      2026|
|   10008|2026-01-22|      2026|
|   10009|2026-01-25|      2026|
|   10010|2026-01-28|      2026|
|   10011|2026-02-02|      2026|
|   10012|2026-02-05|      2026|
|   10013|2026-02-08|      2026|
|   10014|2026-02-11|      2026|
|   10015|2026-02-14|      2026|
|   10016|2026-02-17|      2026|
|   10017|2026-02-20|      2026|
|   10018|2026-02-23|      2026|
|   10019|2026-02-26|      2026|
|   10020|2026-03-01|      2026|
+--------+----------+----------+



Q2. Extract month

In [52]:
q2 = orders_df.select(
    "order_id",
    "order_date",
    month("order_date").alias("order_month"))

q2.show()

+--------+----------+-----------+
|order_id|order_date|order_month|
+--------+----------+-----------+
|   10001|2026-01-05|          1|
|   10002|2026-01-07|          1|
|   10003|2026-01-10|          1|
|   10004|2026-01-12|          1|
|   10005|2026-01-15|          1|
|   10006|2026-01-18|          1|
|   10007|2026-01-20|          1|
|   10008|2026-01-22|          1|
|   10009|2026-01-25|          1|
|   10010|2026-01-28|          1|
|   10011|2026-02-02|          2|
|   10012|2026-02-05|          2|
|   10013|2026-02-08|          2|
|   10014|2026-02-11|          2|
|   10015|2026-02-14|          2|
|   10016|2026-02-17|          2|
|   10017|2026-02-20|          2|
|   10018|2026-02-23|          2|
|   10019|2026-02-26|          2|
|   10020|2026-03-01|          3|
+--------+----------+-----------+



Q3. Find orders placed on weekends

In [54]:
from pyspark.sql.functions import year, month, dayofweek

q3 = orders_df.filter(dayofweek("order_date").isin(1, 7))

q3.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10003|        103|2026-01-10|   Completed|    NetBanking|      Chennai|    Tamil Nadu|     15600.0|          750.0|        200.0|
|   10006|        101|2026-01-18|   Completed|          Card|    Bengaluru|     Karnataka|     22000.0|         1000.0|        250.0|
|   10009|        108|2026-01-25|   Completed|    NetBanking|       Jaipur|     Rajasthan|     18900.0|          900.0|        225.0|
|   10013|        112|2026-02-08|   Completed|    NetBanking|      Kolkata|   West Bengal|     20100.0|          850.0|        210.0|
|   10015|        114|2026-02-14|   Completed|          Card| 

Q4. Find orders placed on Monday

In [55]:
q4 = orders_df.filter( dayofweek("order_date") == 2)

q4.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|
|   10004|        104|2026-01-12|   Cancelled|          Card|       Mumbai|   Maharashtra|      4500.0|          200.0|        120.0|
|   10011|        110|2026-02-02|   Completed|           UPI|    Bengaluru|     Karnataka|     14750.0|          600.0|        160.0|
|   10018|        117|2026-02-23|   Completed|          Card|       Mumbai|   Maharashtra|     24500.0|         1200.0|        275.0|
+--------+-----------+----------+------------+--------------+-

Q5. Calculate order age

In [56]:
q5 = orders_df.select(
    "order_id",
    "order_date",
    datediff(
        current_date(),
        col("order_date")
    ).alias("order_age_days"))

q5.show()

+--------+----------+--------------+
|order_id|order_date|order_age_days|
+--------+----------+--------------+
|   10001|2026-01-05|           265|
|   10002|2026-01-07|           263|
|   10003|2026-01-10|           260|
|   10004|2026-01-12|           258|
|   10005|2026-01-15|           255|
|   10006|2026-01-18|           252|
|   10007|2026-01-20|           250|
|   10008|2026-01-22|           248|
|   10009|2026-01-25|           245|
|   10010|2026-01-28|           242|
|   10011|2026-02-02|           237|
|   10012|2026-02-05|           234|
|   10013|2026-02-08|           231|
|   10014|2026-02-11|           228|
|   10015|2026-02-14|           225|
|   10016|2026-02-17|           222|
|   10017|2026-02-20|           219|
|   10018|2026-02-23|           216|
|   10019|2026-02-26|           213|
|   10020|2026-03-01|           210|
+--------+----------+--------------+



Q6. Find customers registered more than 1 year ago

In [58]:
from pyspark.sql.functions import add_months, col, current_date

q6 = customers_df.filter(
    col("registration_date") < add_months(
        current_date(),
        -12
    )
)

q6.show()

+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|customer_id|first_name|last_name|          email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|
+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|        101|     Rahul|   Sharma|rahul@gmail.com|9876543210|Bangalore|  Karnataka|  India|       2024-01-15|         Premium|   1995-05-12|
|        102|     Priya|    Reddy|priya@gmail.com|9876543211|Hyderabad|  Telangana|  India|       2024-02-20|        Standard|   1997-08-25|
|        103|      Amit|    Kumar| amit@gmail.com|9876543212|  Chennai| Tamil Nadu|  India|       2024-03-10|           Basic|   1994-11-03|
|        104|     Sneha|    Patel|sneha@gmail.com|9876543213|   Mumbai|Maharashtra|  India|       2024-04-05|         Premium|   1996-02-18|
|        105|

Q7. Calculate months since registration

In [65]:
from pyspark.sql.functions import months_between,date_sub,last_day,date_format

q7 = customers_df.select(
    "customer_id",
    "registration_date",
    months_between(
        current_date(),
        col("registration_date")
    ).alias("months_since_registration")
)

q7.show()

+-----------+-----------------+-------------------------+
|customer_id|registration_date|months_since_registration|
+-----------+-----------------+-------------------------+
|        101|       2024-01-15|              32.38709677|
|        102|       2024-02-20|              31.22580645|
|        103|       2024-03-10|               30.5483871|
|        104|       2024-04-05|              29.70967742|
|        105|       2024-05-12|              28.48387097|
+-----------+-----------------+-------------------------+



Q8. Find orders from the last 30 days

In [62]:
q8 = orders_df.filter(
    col("order_date") >= date_sub(
        current_date(),30))

q8.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+



Q9. Find month-end orders

In [64]:
q9 = orders_df.filter( col("order_date") == last_day(col("order_date")))

q9.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+



Q10. Generate YYYY-MM from order date

In [66]:
q10 = orders_df.select(
    "order_id",
    "order_date",
    date_format(
        col("order_date"),
        "yyyy-MM"
    ).alias("year_month")
)

q10.show()

+--------+----------+----------+
|order_id|order_date|year_month|
+--------+----------+----------+
|   10001|2026-01-05|   2026-01|
|   10002|2026-01-07|   2026-01|
|   10003|2026-01-10|   2026-01|
|   10004|2026-01-12|   2026-01|
|   10005|2026-01-15|   2026-01|
|   10006|2026-01-18|   2026-01|
|   10007|2026-01-20|   2026-01|
|   10008|2026-01-22|   2026-01|
|   10009|2026-01-25|   2026-01|
|   10010|2026-01-28|   2026-01|
|   10011|2026-02-02|   2026-02|
|   10012|2026-02-05|   2026-02|
|   10013|2026-02-08|   2026-02|
|   10014|2026-02-11|   2026-02|
|   10015|2026-02-14|   2026-02|
|   10016|2026-02-17|   2026-02|
|   10017|2026-02-20|   2026-02|
|   10018|2026-02-23|   2026-02|
|   10019|2026-02-26|   2026-02|
|   10020|2026-03-01|   2026-03|
+--------+----------+----------+



**Assignment 11** String functions

Q1. Convert names to uppercase

In [68]:
from pyspark.sql.functions import upper, lower, concat, lit, col, length, split, regexp_extract, regexp_replace
q1 = customers_df.select(
    "customer_id",
    "first_name",
    upper("first_name").alias("name_upper"))

q1.show()

+-----------+----------+----------+
|customer_id|first_name|name_upper|
+-----------+----------+----------+
|        101|     Rahul|     RAHUL|
|        102|     Priya|     PRIYA|
|        103|      Amit|      AMIT|
|        104|     Sneha|     SNEHA|
|        105|     Arjun|     ARJUN|
+-----------+----------+----------+



Q2. Convert names to lowercase

In [70]:
q2 = customers_df.select(
    "customer_id",
    "first_name",
    lower("first_name").alias("name_lower")
)

q2.show()

+-----------+----------+----------+
|customer_id|first_name|name_lower|
+-----------+----------+----------+
|        101|     Rahul|     rahul|
|        102|     Priya|     priya|
|        103|      Amit|      amit|
|        104|     Sneha|     sneha|
|        105|     Arjun|     arjun|
+-----------+----------+----------+



Q3. Create full name

In [71]:
q3 = customers_df.select(
    "customer_id",
    "first_name",
    "last_name",
    concat(
        col("first_name"),
        lit(" "),
        col("last_name")
    ).alias("full_name")
)

q3.show()

+-----------+----------+---------+------------+
|customer_id|first_name|last_name|   full_name|
+-----------+----------+---------+------------+
|        101|     Rahul|   Sharma|Rahul Sharma|
|        102|     Priya|    Reddy| Priya Reddy|
|        103|      Amit|    Kumar|  Amit Kumar|
|        104|     Sneha|    Patel| Sneha Patel|
|        105|     Arjun|    Singh| Arjun Singh|
+-----------+----------+---------+------------+



Q4. Find length of customer names

In [73]:
q4 = customers_df.select(
    "customer_id",
    "first_name",
    length("first_name").alias("name_length"))

q4.show()

+-----------+----------+-----------+
|customer_id|first_name|name_length|
+-----------+----------+-----------+
|        101|     Rahul|          5|
|        102|     Priya|          5|
|        103|      Amit|          4|
|        104|     Sneha|          5|
|        105|     Arjun|          5|
+-----------+----------+-----------+



Q5. Extract email username

In [74]:
q5 = customers_df.select(
    "customer_id",
    "email",
    split("email", "@").getItem(0).alias("email_username"))

q5.show()

+-----------+---------------+--------------+
|customer_id|          email|email_username|
+-----------+---------------+--------------+
|        101|rahul@gmail.com|         rahul|
|        102|priya@gmail.com|         priya|
|        103| amit@gmail.com|          amit|
|        104|sneha@gmail.com|         sneha|
|        105|arjun@gmail.com|         arjun|
+-----------+---------------+--------------+



Q6. Extract email domain

In [75]:
q6 = customers_df.select(
    "customer_id",
    "email",
    split("email", "@").getItem(1).alias("email_domain"))

q6.show()

+-----------+---------------+------------+
|customer_id|          email|email_domain|
+-----------+---------------+------------+
|        101|rahul@gmail.com|   gmail.com|
|        102|priya@gmail.com|   gmail.com|
|        103| amit@gmail.com|   gmail.com|
|        104|sneha@gmail.com|   gmail.com|
|        105|arjun@gmail.com|   gmail.com|
+-----------+---------------+------------+



Q7. Extract phone country code

In [76]:
q7 = customers_df.select(
    "customer_id",
    "phone",
    regexp_extract(
        "phone",
        r"^(\+\d+)",
        1
    ).alias("country_code"))

q7.show()

+-----------+----------+------------+
|customer_id|     phone|country_code|
+-----------+----------+------------+
|        101|9876543210|            |
|        102|9876543211|            |
|        103|9876543212|            |
|        104|9876543213|            |
|        105|9876543214|            |
+-----------+----------+------------+



Q8. Remove special characters from phone numbers

In [77]:
q8 = customers_df.select(
    "customer_id",
    "phone",
    regexp_replace(
        "phone",
        r"[^0-9]",
        ""
    ).alias("clean_phone"))

q8.show()

+-----------+----------+-----------+
|customer_id|     phone|clean_phone|
+-----------+----------+-----------+
|        101|9876543210| 9876543210|
|        102|9876543211| 9876543211|
|        103|9876543212| 9876543212|
|        104|9876543213| 9876543213|
|        105|9876543214| 9876543214|
+-----------+----------+-----------+



**Assignment 12** — Other Built-in PySpark Functions

Q1. Replace NULL discount with 0

In [79]:
from pyspark.sql.functions import coalesce, col, lit
q1 = orders_df.select(
    "order_id",
    "discount_amount",
    coalesce(
        col("discount_amount"),
        col("discount_amount").cast("double") * 0
    ).alias("discount_clean"))

q1.show()

+--------+---------------+--------------+
|order_id|discount_amount|discount_clean|
+--------+---------------+--------------+
|   10001|          500.0|         500.0|
|   10002|          250.0|         250.0|
|   10003|          750.0|         750.0|
|   10004|          200.0|         200.0|
|   10005|          400.0|         400.0|
|   10006|         1000.0|        1000.0|
|   10007|          300.0|         300.0|
|   10008|          550.0|         550.0|
|   10009|          900.0|         900.0|
|   10010|          250.0|         250.0|
|   10011|          600.0|         600.0|
|   10012|          350.0|         350.0|
|   10013|          850.0|         850.0|
|   10014|          450.0|         450.0|
|   10015|          650.0|         650.0|
|   10016|          300.0|         300.0|
|   10017|          700.0|         700.0|
|   10018|         1200.0|        1200.0|
|   10019|          350.0|         350.0|
|   10020|          800.0|         800.0|
+--------+---------------+--------

Q2. Replace NULL city with "Unknown"

In [80]:
q2 = customers_df.withColumn(
    "city_clean",
    coalesce(
        col("city"),
        lit("Unknown")))

q2.show()

+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+----------+
|customer_id|first_name|last_name|          email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|city_clean|
+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+----------+
|        101|     Rahul|   Sharma|rahul@gmail.com|9876543210|Bangalore|  Karnataka|  India|       2024-01-15|         Premium|   1995-05-12| Bangalore|
|        102|     Priya|    Reddy|priya@gmail.com|9876543211|Hyderabad|  Telangana|  India|       2024-02-20|        Standard|   1997-08-25| Hyderabad|
|        103|      Amit|    Kumar| amit@gmail.com|9876543212|  Chennai| Tamil Nadu|  India|       2024-03-10|           Basic|   1994-11-03|   Chennai|
|        104|     Sneha|    Patel|sneha@gmail.com|9876543213|   Mumbai|Maharashtra|  Ind

Q3. Create customer category based on spending

In [83]:
from pyspark.sql.functions import when, col, sum

# Assuming customer_spending DataFrame is available from Assignment 6, Task 3 or similar
# If not, it needs to be created first, for example:
# customer_spending = orders_df.groupBy("customer_id").agg(sum("total_amount").alias("total_spending"))

# Join customers_df with customer_spending to get the total_spending for each customer
customer_spending_df_joined = customers_df.join(
    customer_spending,
    customers_df.customer_id == customer_spending.customer_id,
    "left"
).fillna(0, subset=["total_spending"]) # Handle customers with no orders


q3 = customer_spending_df_joined.withColumn(
    "customer_category",
    when(col("total_spending") >= 50000, "Premium")
    .when(col("total_spending") >= 20000, "Gold")
    .otherwise("Standard"))

q3.show()

+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+-----------+--------------+-----------------+
|customer_id|first_name|last_name|          email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|customer_id|total_spending|customer_category|
+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+-----------+--------------+-----------------+
|        101|     Rahul|   Sharma|rahul@gmail.com|9876543210|Bangalore|  Karnataka|  India|       2024-01-15|         Premium|   1995-05-12|        101|       34500.0|             Gold|
|        102|     Priya|    Reddy|priya@gmail.com|9876543211|Hyderabad|  Telangana|  India|       2024-02-20|        Standard|   1997-08-25|        102|        8200.0|         Standard|
|        103|      Amit|    Kumar| amit@gmail.com|9876543212|  Chennai

Q4. Round revenue to 2 decimal places

In [85]:
from pyspark.sql.functions import round, col

q4 = orders_df.withColumn(
    "revenue_rounded",
    round(col("total_amount"), 2))

q4.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|revenue_rounded|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+---------------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|        12500.0|
|   10002|        102|2026-01-07|     Pending|           UPI|    Hyderabad|     Telangana|      8200.0|          250.0|        100.0|         8200.0|
|   10003|        103|2026-01-10|   Completed|    NetBanking|      Chennai|    Tamil Nadu|     15600.0|          750.0|        200.0|        15600.0|
|   10004|        104|2026-01-12|   Cancelled|          Card|       Mumbai|   Maharashtra|      4500

Q5. Calculate profit

In [87]:
q5 = orders_df.withColumn("profit",round(col("total_amount") - (col("discount_amount") + col("shipping_cost")), 2))
q5.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+-------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost| profit|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+-------+
|   10001|        101|2026-01-05|   Completed|          Card|    Bengaluru|     Karnataka|     12500.0|          500.0|        150.0|11850.0|
|   10002|        102|2026-01-07|     Pending|           UPI|    Hyderabad|     Telangana|      8200.0|          250.0|        100.0| 7850.0|
|   10003|        103|2026-01-10|   Completed|    NetBanking|      Chennai|    Tamil Nadu|     15600.0|          750.0|        200.0|14650.0|
|   10004|        104|2026-01-12|   Cancelled|          Card|       Mumbai|   Maharashtra|      4500.0|          200.0|        120.0| 4180.0|
|   10

In [95]:
from pyspark.sql.functions import array, col

products_array_df = products_df.withColumn(
    "product_array",
    array(
        col("product_name"),
        col("category"),
        col("brand")
    )
)

products_array_df.show(truncate=False)

+----------+-----------------------+---------------+----------------+-----------+-----------+---------+-------------+-----------+--------------+--------------+---------------------------------------------------+
|product_id|product_name           |category       |sub_category    |brand      |supplier_id|unit_cost|selling_price|launch_date|product_rating|stock_quantity|product_array                                      |
+----------+-----------------------+---------------+----------------+-----------+-----------+---------+-------------+-----------+--------------+--------------+---------------------------------------------------+
|501       |Laptop Pro 14          |Electronics    |Laptops         |TechNova   |201        |52000.0  |68000.0      |2025-01-15 |4.5           |120           |[Laptop Pro 14, Electronics, TechNova]             |
|502       |Wireless Headphones    |Electronics    |Audio           |SoundMax   |202        |1800.0   |2500.0       |2025-02-10 |4.3           |250     

Q7. Explode product arrays

In [96]:
from pyspark.sql.functions import explode

q7 = products_array_df.select(
    "product_id",
    explode("product_array").alias("product_value")
)

q7.show(truncate=False)


+----------+-------------------+
|product_id|product_value      |
+----------+-------------------+
|501       |Laptop Pro 14      |
|501       |Electronics        |
|501       |TechNova           |
|502       |Wireless Headphones|
|502       |Electronics        |
|502       |SoundMax           |
|503       |Mechanical Keyboard|
|503       |Electronics        |
|503       |KeyCraft           |
|504       |4K Monitor 27      |
|504       |Electronics        |
|504       |ViewTech           |
|505       |USB-C Hub          |
|505       |Electronics        |
|505       |ConnectPro         |
|506       |Smart Watch X1     |
|506       |Electronics        |
|506       |FitPulse           |
|507       |Running Shoes      |
|507       |Fashion            |
+----------+-------------------+
only showing top 20 rows


In [91]:
products_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- unit_cost: double (nullable = true)
 |-- selling_price: double (nullable = true)
 |-- launch_date: date (nullable = true)
 |-- product_rating: double (nullable = true)
 |-- stock_quantity: integer (nullable = true)



Q8. Find array length

In [97]:
from pyspark.sql.functions import size

q8 = products_array_df.select(
    "product_id",
    "product_array",
    size("product_array").alias("array_length")
)

q8.show(truncate=False)

+----------+---------------------------------------------------+------------+
|product_id|product_array                                      |array_length|
+----------+---------------------------------------------------+------------+
|501       |[Laptop Pro 14, Electronics, TechNova]             |3           |
|502       |[Wireless Headphones, Electronics, SoundMax]       |3           |
|503       |[Mechanical Keyboard, Electronics, KeyCraft]       |3           |
|504       |[4K Monitor 27, Electronics, ViewTech]             |3           |
|505       |[USB-C Hub, Electronics, ConnectPro]               |3           |
|506       |[Smart Watch X1, Electronics, FitPulse]            |3           |
|507       |[Running Shoes, Fashion, SportEdge]                |3           |
|508       |[Gaming Laptop, Electronics, GameCore]             |3           |
|509       |[Office Chair, Furniture, ComfortSeat]             |3           |
|510       |[Smartphone A15, Electronics, MobileMax]           |

Q9. Check whether an array contains a specific value

In [98]:
from pyspark.sql.functions import array_contains

q9 = products_array_df.select(
    "product_id",
    "product_array",
    array_contains(
        "product_array",
        "Electronics"
    ).alias("contains_electronics")
)

q9.show(truncate=False)

+----------+---------------------------------------------------+--------------------+
|product_id|product_array                                      |contains_electronics|
+----------+---------------------------------------------------+--------------------+
|501       |[Laptop Pro 14, Electronics, TechNova]             |true                |
|502       |[Wireless Headphones, Electronics, SoundMax]       |true                |
|503       |[Mechanical Keyboard, Electronics, KeyCraft]       |true                |
|504       |[4K Monitor 27, Electronics, ViewTech]             |true                |
|505       |[USB-C Hub, Electronics, ConnectPro]               |true                |
|506       |[Smart Watch X1, Electronics, FitPulse]            |true                |
|507       |[Running Shoes, Fashion, SportEdge]                |false               |
|508       |[Gaming Laptop, Electronics, GameCore]             |true                |
|509       |[Office Chair, Furniture, ComfortSeat]    

**Assignment 13 **— UDF

In [102]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

# Q1 - Normal Python function
def get_country_name(country_code):
    country_map = {
        "IN": "India",
        "US": "United States",
        "UK": "United Kingdom",
        "AU": "Australia",
        "SG": "Singapore"
    }

    return country_map.get(country_code, "Unknown")

def get_country_name(country_code):

    country_map = {
        "IN": "India",
        "US": "United States",
        "UK": "United Kingdom",
        "AU": "Australia",
        "SG": "Singapore"
    }

    return country_map.get(country_code, "Unknown")

Q2. Convert it into a PySpark UDF

In [103]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def get_country_name(country_code):

    country_map = {
        "IN": "India",
        "US": "United States",
        "UK": "United Kingdom",
        "AU": "Australia",
        "SG": "Singapore"
    }

    return country_map.get(country_code, "Unknown")

country_name_udf = udf(
    get_country_name,
    StringType()
)

Q3. Apply it to a DataFrame

In [104]:
customers_with_country = customers_df.withColumn(
    "country_name",
    country_name_udf(
        col("country")))

customers_with_country.show()

+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+------------+
|customer_id|first_name|last_name|          email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|country_name|
+-----------+----------+---------+---------------+----------+---------+-----------+-------+-----------------+----------------+-------------+------------+
|        101|     Rahul|   Sharma|rahul@gmail.com|9876543210|Bangalore|  Karnataka|  India|       2024-01-15|         Premium|   1995-05-12|     Unknown|
|        102|     Priya|    Reddy|priya@gmail.com|9876543211|Hyderabad|  Telangana|  India|       2024-02-20|        Standard|   1997-08-25|     Unknown|
|        103|      Amit|    Kumar| amit@gmail.com|9876543212|  Chennai| Tamil Nadu|  India|       2024-03-10|           Basic|   1994-11-03|     Unknown|
|        104|     Sneha|    Patel|sneha@gmail.com|9876543213|   Mumbai|Mahar

Q4. Create a UDF that categorizes customers based on age

In [105]:
def categorize_age(age):

    if age is None:
        return "Unknown"

    if age < 18:
        return "Minor"

    elif age < 60:
        return "Adult"

    else:
        return "Senior"

Q5. Create a UDF that categorizes orders based on amount

In [106]:
def categorize_order_amount(amount):

    if amount is None:
        return "Unknown"

    if amount < 1000:
        return "Low"

    elif amount < 5000:
        return "Medium"

    else:
        return "High"

Q6. Create a UDF that masks phone numbers

In [107]:
def mask_phone(phone):

    if phone is None:
        return None

    phone = str(phone)

    if len(phone) <= 4:
        return "*" * len(phone)

    return "*" * (len(phone) - 4) + phone[-4:]

**Assignment 14** — cache() and persist()

Q1. What is lazy evaluation?

Answer:
Lazy evaluation means PySpark does not execute transformations immediately when they are written.

Transformations such as filter(), select(), groupBy(), and join() build an execution plan. Spark executes that plan only when an action such as show(), count(), collect(), or writing data is called.

Example concept:

Q2. Why does cache help?

Answer:
Caching stores the computed result of a DataFrame so that it can be reused without recomputing the entire transformation chain every time.

It is especially useful when the same DataFrame is used multiple times.

Q3. When should we cache?

We should consider caching when:

A DataFrame is reused multiple times.
The transformation used to create the DataFrame is expensive.
The dataset is used for multiple actions or analyses.
We are performing interactive analysis and repeatedly accessing the same transformed data.

Q4. When should we NOT cache?

Answer:

We should avoid caching when:

The DataFrame is used only once.
The dataset is extremely large and caching consumes excessive resources.
The transformation is simple and inexpensive to recompute.
Memory/storage resources are limited.
The cached DataFrame is unlikely to be reused.

Q5. What is the difference between cache() and persist()?

Answer:

Both are used to keep a DataFrame's computed data available for reuse.

cache()

Simple way to request the default persistence behavior.
Used when you don't need to specify a storage level.

persist()

Gives more control.
Allows you to specify a particular storage level such as memory or memory plus disk.

Q6. What happens when the cached data does not fit in memory?

Answer:

It depends on the storage level being used.

If the data is persisted using a memory-and-disk storage level, Spark keeps the data that fits in memory and stores the remaining data on disk.

Conceptually: